[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/legalintermedia/Physics/blob/main/Simulaciones_Jupyter/Investigacion_Avanzada/Relatividad_Kerr_Ray_Tracing.ipynb)



# Investigación Avanzada: Relatividad Kerr Ray Tracing

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def kerr_geodesic_derivatives(tau, y, M, a):
    """
    Geodesic equations for a photon in the equatorial plane of a Kerr black hole.
    y = [t, r, phi, p_t, p_r, p_phi]
    """
    t, r, phi, pt, pr, pphi = y
    
    # Metric components in equatorial plane (theta = pi/2)
    Sigma = r**2
    Delta = r**2 - 2*M*r + a**2
    
    # Contravariant metric components
    g_tt = -(1 + 2*M*r/Sigma)
    g_tphi = -2*M*a*r/Sigma
    g_phiphi = (r**2 + a**2 + 2*M*a**2*r/Sigma) * np.sin(np.pi/2)**2
    g_rr = Sigma/Delta
    
    # Inverse metric components
    det = g_tt*g_phiphi - g_tphi**2
    g_up_tt = g_phiphi / det
    g_up_tphi = -g_tphi / det
    g_up_phiphi = g_tt / det
    g_up_rr = 1.0 / g_rr
    
    # Velocities (dot{x} = dx/dtau)
    tdot = g_up_tt * pt + g_up_tphi * pphi
    phidot = g_up_tphi * pt + g_up_phiphi * pphi
    rdot = g_up_rr * pr
    
    # Christoffel symbols derivatives for momenta (Hamilton's equations)
    # d p_mu / dtau = - 1/2 (d g^{alpha beta} / d x^mu) p_alpha p_beta
    # For a photon, p_t = -E (constant), p_phi = L (constant)
    # So pt_dot = 0, pphi_dot = 0
    pt_dot = 0.0
    pphi_dot = 0.0
    
    # p_r dot equation from H = 1/2 g^{mu nu} p_mu p_nu = 0
    # Actually simpler: H = 0 implies (p_r)^2 can be found. 
    # Let's directly differentiate the effective potential.
    # dpr/dtau = -1/2 d/dr (g^{tt} pt^2 + 2 g^{tphi} pt pphi + g^{phiphi} pphi^2 + g^{rr} pr^2)
    # Numerical derivative of the metric components
    dr = 1e-5
    Sigma_dr = (r+dr)**2
    Delta_dr = (r+dr)**2 - 2*M*(r+dr) + a**2
    g_tt_dr = -(1 + 2*M*(r+dr)/Sigma_dr)
    g_tphi_dr = -2*M*a*(r+dr)/Sigma_dr
    g_phiphi_dr = ((r+dr)**2 + a**2 + 2*M*a**2*(r+dr)/Sigma_dr)
    g_rr_dr = Sigma_dr/Delta_dr
    
    det_dr = g_tt_dr*g_phiphi_dr - g_tphi_dr**2
    g_up_tt_dr = g_phiphi_dr / det_dr
    g_up_tphi_dr = -g_tphi_dr / det_dr
    g_up_phiphi_dr = g_tt_dr / det_dr
    g_up_rr_dr = 1.0 / g_rr_dr
    
    dH_dr = 0.5 * ( (g_up_tt_dr - g_up_tt)/dr * pt**2 + 
                    2 * (g_up_tphi_dr - g_up_tphi)/dr * pt * pphi + 
                    (g_up_phiphi_dr - g_up_phiphi)/dr * pphi**2 + 
                    (g_up_rr_dr - g_up_rr)/dr * pr**2 )
    
    pr_dot = -dH_dr
    
    return [tdot, rdot, phidot, pt_dot, pr_dot, pphi_dot]

def simulate_ray_tracing():
    M = 1.0
    a = 0.95 # highly spinning
    
    # Photon initial conditions
    # Far away photon with impact parameter b
    b_values = np.linspace(-7, 7, 30)
    
    plt.figure(figsize=(10, 10))
    # Event horizon
    r_H = M + np.sqrt(M**2 - a**2)
    circle = plt.Circle((0, 0), r_H, color='black', zorder=10)
    plt.gca().add_patch(circle)
    
    # Ergosphere in equatorial plane
    r_E = 2*M
    circle_E = plt.Circle((0, 0), r_E, color='gray', fill=False, linestyle='--', zorder=9)
    plt.gca().add_patch(circle_E)
    
    for b in b_values:
        r0 = 20.0
        phi0 = 0.0
        t0 = 0.0
        
        pt = -1.0 # E = 1
        pphi = b  # L = b
        
        # Calculate pr from H = 0
        Sigma = r0**2
        Delta = r0**2 - 2*M*r0 + a**2
        g_tt = -(1 + 2*M*r0/Sigma)
        g_tphi = -2*M*a*r0/Sigma
        g_phiphi = (r0**2 + a**2 + 2*M*a**2*r0/Sigma)
        g_rr = Sigma/Delta
        
        det = g_tt*g_phiphi - g_tphi**2
        g_up_tt = g_phiphi / det
        g_up_tphi = -g_tphi / det
        g_up_phiphi = g_tt / det
        g_up_rr = 1.0 / g_rr
        
        # H = 1/2 (g^{tt} pt^2 + 2 g^{tphi} pt pphi + g^{phiphi} pphi^2 + g^{rr} pr^2) = 0
        pr2 = -(g_up_tt * pt**2 + 2 * g_up_tphi * pt * pphi + g_up_phiphi * pphi**2) / g_up_rr
        if pr2 < 0:
            continue
        pr = -np.sqrt(pr2) # incoming photon
        
        y0 = [t0, r0, phi0, pt, pr, pphi]
        tau_span = (0, 40)
        
        # Stop integration if photon falls into black hole (r < r_H)
        def hit_BH(t, y, M, a):
            return y[1] - r_H*1.01
        hit_BH.terminal = True
        
        sol = solve_ivp(kerr_geodesic_derivatives, tau_span, y0, args=(M, a), 
                        dense_output=True, events=hit_BH, rtol=1e-8, atol=1e-8)
        
        r_sol = sol.y[1]
        phi_sol = sol.y[2]
        
        x = r_sol * np.cos(phi_sol)
        y = r_sol * np.sin(phi_sol)
        plt.plot(x, y, color='orange' if b > 0 else 'blue', alpha=0.7)

    plt.xlim(-10, 20)
    plt.ylim(-10, 10)
    plt.aspect='equal'
    plt.title(f'Kerr Black Hole Ray Tracing (Equatorial Plane), a={a}')
    plt.xlabel('x (GM/c^2)')
    plt.ylabel('y (GM/c^2)')
    plt.grid(True, alpha=0.3)
    plt.savefig('Kerr_Ray_Tracing.png', dpi=300)
    print("Simulation complete. Image saved as Kerr_Ray_Tracing.png")

if __name__ == '__main__':
    simulate_ray_tracing()
